In [94]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import VotingRegressor
from sklearn.model_selection import cross_val_score
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import cross_val_score
train_data = pd.read_csv(r"C:\Users\chall\Downloads\rcbp_2\train.csv")
test_data = pd.read_csv(r"C:\Users\chall\Downloads\rcbp_2\test.csv") 

In [95]:
missing_values = train_data.isna().sum()
print(missing_values)
print(train_data.dtypes)

Id                        0
BuiltArea                 2
LotArea                 252
Bedrooms                  0
Bathrooms                 1
Floors                    1
Age                       0
DistanceToCityCenter      1
SchoolRating            250
CrimeIndex                1
HasParking                0
HasGym                    0
HasSwimmingPool           0
FurnishingType            1
SalePrice                 0
dtype: int64
Id                        int64
BuiltArea               float64
LotArea                 float64
Bedrooms                  int64
Bathrooms               float64
Floors                  float64
Age                       int64
DistanceToCityCenter    float64
SchoolRating            float64
CrimeIndex              float64
HasParking                int64
HasGym                    int64
HasSwimmingPool           int64
FurnishingType           object
SalePrice               float64
dtype: object


In [96]:
train_data.head()

,Id,BuiltArea,LotArea,Bedrooms,Bathrooms,Floors,Age,DistanceToCityCenter,SchoolRating,CrimeIndex,HasParking,HasGym,HasSwimmingPool,FurnishingType,SalePrice
0,1,1748.357077,1660.992254,2,2.0,3.0,25,3.685357,NaN,56.348909,0,1,0,Semi-Furnished,124.766616
1,2,1430.867849,1637.268713,1,4.0,3.0,2,9.040218,10.0,30.723171,0,0,0,Fully-Furnished,126.024057
2,3,1823.844269,563.485462,5,4.0,3.0,1,11.600823,9.0,40.084497,1,0,0,Unfurnished,115.837455
3,4,2261.514928,1735.927847,2,2.0,2.0,4,2.102469,1.0,43.413042,0,1,0,Unfurnished,145.638087
4,5,1382.923313,2586.263265,1,1.0,2.0,19,18.054574,3.0,35.274963,1,1,0,NaN,91.438792


# Figuring out what to do with the missing values

we fill the missing values with Median Values while doing a function



In [97]:
def features_missing_values (df):
    df = df.copy()

    #solving missing values specific to area
    Built_Area = df['BuiltArea'].fillna(df['BuiltArea'].median())
    Lot_Area = df['LotArea'].fillna(df['LotArea'].median())

    # Bulding feature, which helps it easier to predict prices
    df['Total_Area'] = Built_Area + Lot_Area

    ### Only Keep them if RMSE improves, remove otherwise
    
    # Area per Bedroom (+1 to prevent division by zero)
    df['Area_Bedroom'] = Built_Area / (df['Bedrooms'] + 1) 
    
    # Overall Amenities Quality proxy
    df['AmenitiesCount'] = df['HasParking'] + df['HasGym'] + df['HasSwimmingPool']
    
    return df



In [98]:
# applying the features of the Missing values 
X_train = features_missing_values(train_data).drop(['Id', 'SalePrice'], axis = 1) # Independent variable
y_train = train_data['SalePrice'] # Output 
X_test = features_missing_values(test_data).drop(['Id'], axis = 1)

In [99]:
numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X_train.select_dtypes(include=['object']).columns

In [100]:
# automating the fixing of missing values for other data and scaling to better suit the regression analysis
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
# with the most frequent category and converting them into numerical format 
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))# using one-hot encoding; also applies separate preprocessing steps to 
])
# numeric and categorical features before model training
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

In [3]:
# Define individual models
model_xgb = xgb.XGBRegressor(
    n_estimators=600,
    learning_rate=0.04,
    max_depth=5,
    random_state=42
)

model_lgb = lgb.LGBMRegressor(
    n_estimators=600,
    learning_rate=0.04,
    max_depth=5,
    random_state=42,
    verbose=-1
)

NameError: name 'xgb' is not defined

In [102]:
ensemble_model = VotingRegressor(
    estimators=[('xgb', model_xgb), ('lgb', model_lgb)]
)
#averages predictions from both models.

In [103]:
# Log Transform Target
log_target_ensemble = TransformedTargetRegressor(
    regressor=ensemble_model,
    func=np.log1p,
    inverse_func=np.expm1
)
#improves RMSE significantly.


In [104]:
# Bulding Pipeline
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', log_target_ensemble)
])

In [105]:
full_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['BuiltArea', 'LotArea', 'Bedrooms', 'Bathrooms', 'Floors', 'Age',
       'DistanceToCityCenter', 'SchoolRating', 'CrimeIndex', 'HasParking',
       'HasGym', 'HasSwimmingPool', 'Total_Area', 'Area_Bedroom',
       'A...
                                                                                                learning_rate=0.04,
                                                                                                max_bin=None,
                                                                                                max_cat_threshold=None,
                                                                                                max_cat_to_onehot=None,
                                                                                                max_delta_step=None,
                                                                                                max_depth=5,
                                                                                                max_leaves=None,
                                                                                                min_child_weight=None,
                                                                                                missing=nan,
                                                                                                monotone_constraints=None,
                                                                                                multi_strategy=None,
                                                                                                n_estimators=600,
                                                                                                n_jobs=None,
                                                                                                num_parallel_tree=None, ...)),
                                                                                  ('lgb',
                                                                                   LGBMRegressor(learning_rate=0.04,
                                                                                                 max_depth=5,
                                                                                                 n_estimators=600,
                                                                                                 random_state=42,
                                                                                                 verbose=-1))])))])

In [106]:

rmse_scores = -cross_val_score(
    full_pipeline,
    X_train,
    y_train,
    scoring="neg_root_mean_squared_error",
    cv=5
)

print("RMSE scores:", rmse_scores)
print("Mean RMSE:", rmse_scores.mean())

C:\Users\chall\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\chall\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\chall\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\chall\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


RMSE scores: [16.17726002 16.36351354 15.81342896 16.11964252 15.60740072]
Mean RMSE: 16.016249150496243


C:\Users\chall\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [78]:
preds = full_pipeline.predict(X_test)

C:\Users\chall\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [89]:
# Create submission file
submission = pd.DataFrame({
    'Id': test_data['Id'],
    'SalePrice': preds
})
submission.to_csv("submission.csv", index=False)